# Recommendation Baselines — Phase 3

This notebook implements Phase 3 steps 1–5: final artifact-contract validation, task definitions, candidate policies and ranking metrics. It deliberately does not implement recommendation baselines yet; those begin at step 6.

> Validation-future data is evaluation-only. Discovery-future labels may be used for model training in later steps, but no validation-future label may influence fitting or tuning.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1 — Imports, paths and immutable experiment configuration

In [2]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BENCHMARK_ROOT = Path('/content/drive/MyDrive/datasets/recommendation_benchmark_final_outputs')
OUTPUT_ROOT = Path('/content/drive/MyDrive/datasets/recommendation_baseline_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_DIRS = {
    'ASSISTments': BENCHMARK_ROOT / 'assistments',
    'KDD': BENCHMARK_ROOT / 'kdd',
}
METRIC_KS = (5, 10, 20)
PRIMARY_RELEVANCE = 'attempted'
RANDOM_STATE = 42

ROOT_FILES = [
    'benchmark_summary.csv', 'artifact_manifest.csv',
    'benchmark_config.json', 'benchmark_schema.json',
]
for filename in ROOT_FILES:
    path = BENCHMARK_ROOT / filename


## Step 2 — Validate the 18-artifact benchmark contract

This validation uses Parquet metadata wherever possible so the four-million-row ASSISTments histories are not loaded merely to check their schemas. Small learner and catalogue tables are loaded for semantic assertions.

In [3]:
with open(BENCHMARK_ROOT / 'benchmark_config.json', encoding='utf-8') as file:
    benchmark_config = json.load(file)
with open(BENCHMARK_ROOT / 'benchmark_schema.json', encoding='utf-8') as file:
    benchmark_schema = json.load(file)
benchmark_summary = pd.read_csv(BENCHMARK_ROOT / 'benchmark_summary.csv')
phase2_manifest = pd.read_csv(BENCHMARK_ROOT / 'artifact_manifest.csv')

assert benchmark_config['candidate_statistics_source'] == 'discovery_early_only'
assert benchmark_config['future_role'] == 'relevance_labels_only'
assert benchmark_config['assist_skill_name_mapping_source'] == 'discovery_early_only'
assert len(benchmark_schema) == 9
assert len(phase2_manifest) == 18
assert set(phase2_manifest['Dataset']) == set(DATASET_DIRS)

REQUIRED_COLUMNS = {
    'learner_splits.parquet': {
        'learner_id', 'cohort', 'evaluable_skill', 'evaluable_problem',
        'cold_start_skill_history', 'cold_start_problem_history',
        'cluster', 'cluster_selection_status', 'cluster_allowed_as_primary',
    },
    'skill_catalog.parquet': {
        'item_id', 'item_type', 'training_interactions', 'training_learners',
        'training_success_rate', 'training_popularity', 'catalog_source',
    },
    'problem_catalog.parquet': {
        'item_id', 'item_type', 'training_interactions', 'training_learners',
        'training_success_rate', 'training_popularity', 'catalog_source',
        'problem_type', 'hierarchy',
    },
    'early_skill_history.parquet': {
        'learner_id', 'item_id', 'early_interaction_count',
        'empirical_bayes_mastery', 'mastery_evidence_confidence',
        'skill_state', 'in_candidate_catalog',
    },
    'future_skill_relevance.parquet': {
        'learner_id', 'item_id', 'relevance_binary',
        'successful_future_item', 'in_candidate_catalog', 'seen_in_early',
    },
    'early_problem_history.parquet': {
        'learner_id', 'item_id', 'early_interaction_count',
        'early_success_rate', 'in_candidate_catalog',
    },
    'future_problem_relevance.parquet': {
        'learner_id', 'item_id', 'relevance_binary',
        'successful_future_item', 'in_candidate_catalog', 'seen_in_early',
    },
    'problem_skill_map.parquet': {
        'problem_item_id', 'skill_item_id', 'training_interactions',
        'association_share', 'mapping_source',
    },
    'skill_name_id_map.parquet': {
        'normalised_skill_name', 'mapped_skill_id', 'training_rows',
        'training_learners', 'distinct_skill_ids', 'mapping_source',
    },
}
assert set(REQUIRED_COLUMNS) == set(benchmark_schema)

contract_rows = []
small_tables = {}
for dataset, directory in DATASET_DIRS.items():
    dataset_manifest = phase2_manifest[phase2_manifest['Dataset'].eq(dataset)].set_index('File')
    assert set(dataset_manifest.index) == set(REQUIRED_COLUMNS)
    for filename, required_columns in REQUIRED_COLUMNS.items():
        path = directory / filename
        parquet_file = pq.ParquetFile(path)
        row_count = parquet_file.metadata.num_rows
        schema_columns = set(parquet_file.schema_arrow.names)
        missing_columns = sorted(required_columns - schema_columns)
        manifest_row = dataset_manifest.loc[filename]
        assert row_count == int(manifest_row['Rows'])
        assert path.stat().st_size == int(manifest_row['Bytes'])
        assert not missing_columns, f'{dataset}/{filename}: {missing_columns}'
        contract_rows.append({
            'Dataset': dataset, 'File': filename, 'Rows': row_count,
            'Bytes': path.stat().st_size, 'SchemaValid': True,
            'ManifestValid': True,
        })

    learners = pd.read_parquet(directory / 'learner_splits.parquet')
    skill_catalog = pd.read_parquet(directory / 'skill_catalog.parquet')
    problem_catalog = pd.read_parquet(directory / 'problem_catalog.parquet')
    problem_skill_map = pd.read_parquet(directory / 'problem_skill_map.parquet')
    skill_name_map = pd.read_parquet(directory / 'skill_name_id_map.parquet')
    assert set(learners['cohort']) == {'discovery', 'validation'}
    assert learners['cluster'].notna().all()
    assert skill_catalog['catalog_source'].eq('discovery_early_only').all()
    assert problem_catalog['catalog_source'].eq('discovery_early_only').all()
    assert problem_skill_map['mapping_source'].eq('discovery_early_only').all()
    if len(skill_name_map):
        assert skill_name_map['mapping_source'].eq('discovery_early_only').all()
    small_tables[dataset] = {
        'learners': learners, 'skill_catalog': skill_catalog,
        'problem_catalog': problem_catalog,
    }

assert small_tables['ASSISTments']['learners']['cluster_selection_status'].eq('accepted').all()
assert small_tables['KDD']['learners']['cluster_selection_status'].eq('exploratory_fallback').all()
assert not small_tables['KDD']['learners']['cluster_allowed_as_primary'].any()
contract_validation = pd.DataFrame(contract_rows)
display(benchmark_summary)
display(contract_validation)
print('Validated 18 Phase 2 Parquet artifacts without loading the large histories.')

,Dataset,RawRows,RawLearners,EligibleLearners,DiscoveryLearners,ValidationLearners,EarlyInteractions,FutureInteractions,SkillCandidates,ProblemCandidates,...,ValidationProblemEvaluableRate,ValidationSkillColdStartRate,ValidationProblemColdStartRate,ClusterSelectionStatus,TrainingSkillNameMappings,AmbiguousTrainingSkillNames,EarlyRowsMappedFromName,FutureRowsMappedFromName,EarlyTextFallbackRows,FutureTextFallbackRows
0,ASSISTments,6117947,46667,33335,26668,6667,4184236,1814850,161,39779,...,0.899355,0.375731,0.024599,accepted,194,33,0,0,0,0
1,KDD,809694,574,565,452,113,566460,243152,99,855,...,0.982301,0.000000,0.000000,exploratory_fallback,0,0,0,0,0,0


,Dataset,File,Rows,Bytes,SchemaValid,ManifestValid
0,ASSISTments,learner_splits.parquet,33335,1283461,True,True
1,ASSISTments,skill_catalog.parquet,161,16637,True,True
2,ASSISTments,problem_catalog.parquet,39779,978772,True,True
3,ASSISTments,early_skill_history.parquet,260768,8707997,True,True
4,ASSISTments,future_skill_relevance.parquet,174823,4960293,True,True
5,ASSISTments,early_problem_history.parquet,4052498,76911901,True,True
6,ASSISTments,future_problem_relevance.parquet,1783782,35311927,True,True
7,ASSISTments,problem_skill_map.parquet,17826,176578,True,True
8,ASSISTments,skill_name_id_map.parquet,194,10524,True,True
9,KDD,learner_splits.parquet,565,46435,True,True


Validated 18 Phase 2 Parquet artifacts without loading the large histories.


## Step 3 — Define datasets, tasks and relevance labels

Later attempted items are the primary offline target. Later successful items are a sensitivity target. Neither target is interpreted as causal learning benefit.

In [4]:
RELEVANCE_DEFINITIONS = {
    'attempted': 'relevance_binary',
    'successful': 'successful_future_item',
}
TASK_DEFINITIONS = [
    {
        'Dataset': 'ASSISTments', 'Task': 'problem', 'Priority': 'primary',
        'CatalogFile': 'problem_catalog.parquet',
        'EarlyHistoryFile': 'early_problem_history.parquet',
        'FutureRelevanceFile': 'future_problem_relevance.parquet',
        'EvaluableColumn': 'evaluable_problem',
        'ColdStartColumn': 'cold_start_problem_history',
        'ClusterUse': 'accepted_ablation',
    },
    {
        'Dataset': 'ASSISTments', 'Task': 'skill', 'Priority': 'secondary',
        'CatalogFile': 'skill_catalog.parquet',
        'EarlyHistoryFile': 'early_skill_history.parquet',
        'FutureRelevanceFile': 'future_skill_relevance.parquet',
        'EvaluableColumn': 'evaluable_skill',
        'ColdStartColumn': 'cold_start_skill_history',
        'ClusterUse': 'accepted_ablation',
    },
    {
        'Dataset': 'KDD', 'Task': 'problem', 'Priority': 'external_validation',
        'CatalogFile': 'problem_catalog.parquet',
        'EarlyHistoryFile': 'early_problem_history.parquet',
        'FutureRelevanceFile': 'future_problem_relevance.parquet',
        'EvaluableColumn': 'evaluable_problem',
        'ColdStartColumn': 'cold_start_problem_history',
        'ClusterUse': 'exploratory_ablation_only',
    },
    {
        'Dataset': 'KDD', 'Task': 'skill', 'Priority': 'external_validation',
        'CatalogFile': 'skill_catalog.parquet',
        'EarlyHistoryFile': 'early_skill_history.parquet',
        'FutureRelevanceFile': 'future_skill_relevance.parquet',
        'EvaluableColumn': 'evaluable_skill',
        'ColdStartColumn': 'cold_start_skill_history',
        'ClusterUse': 'exploratory_ablation_only',
    },
]
task_table = pd.DataFrame(TASK_DEFINITIONS)
summary_lookup = benchmark_summary.set_index('Dataset')
task_table['CandidateCount'] = task_table.apply(
    lambda row: int(summary_lookup.loc[row['Dataset'], 'ProblemCandidates' if row['Task'] == 'problem' else 'SkillCandidates']),
    axis=1,
)
task_table['ValidationEvaluableRate'] = task_table.apply(
    lambda row: float(summary_lookup.loc[row['Dataset'], 'ValidationProblemEvaluableRate' if row['Task'] == 'problem' else 'ValidationSkillEvaluableRate']),
    axis=1,
)
task_table['ValidationColdStartRate'] = task_table.apply(
    lambda row: float(summary_lookup.loc[row['Dataset'], 'ValidationProblemColdStartRate' if row['Task'] == 'problem' else 'ValidationSkillColdStartRate']),
    axis=1,
)
assert len(task_table) == 4
assert task_table['CandidateCount'].gt(0).all()
assert set(RELEVANCE_DEFINITIONS) == {'attempted', 'successful'}
display(task_table)

,Dataset,Task,Priority,CatalogFile,EarlyHistoryFile,FutureRelevanceFile,EvaluableColumn,ColdStartColumn,ClusterUse,CandidateCount,ValidationEvaluableRate,ValidationColdStartRate
0,ASSISTments,problem,primary,problem_catalog.parquet,early_problem_history.parquet,future_problem_relevance.parquet,evaluable_problem,cold_start_problem_history,accepted_ablation,39779,0.899355,0.024599
1,ASSISTments,skill,secondary,skill_catalog.parquet,early_skill_history.parquet,future_skill_relevance.parquet,evaluable_skill,cold_start_skill_history,accepted_ablation,161,0.588271,0.375731
2,KDD,problem,external_validation,problem_catalog.parquet,early_problem_history.parquet,future_problem_relevance.parquet,evaluable_problem,cold_start_problem_history,exploratory_ablation_only,855,0.982301,0.000000
3,KDD,skill,external_validation,skill_catalog.parquet,early_skill_history.parquet,future_skill_relevance.parquet,evaluable_skill,cold_start_skill_history,exploratory_ablation_only,99,0.991150,0.000000


## Step 4 — Candidate policies

Candidate-policy filtering operates on already-generated sparse score rows. It never creates a dense learner-by-catalogue cross-product.

In [5]:
CANDIDATE_POLICIES = {
    'all_supported': {
        'exclude_seen': False,
        'description': 'Allow supported items seen previously; repeated practice is valid.',
    },
    'novel_only': {
        'exclude_seen': True,
        'description': 'Exclude items present in the learner early history.',
    },
}

def apply_candidate_policy(scored_candidates, early_history, policy):
    if policy not in CANDIDATE_POLICIES:
        raise ValueError(f'Unknown candidate policy: {policy}')
    required = {'learner_id', 'item_id', 'score'}
    if not required.issubset(scored_candidates.columns):
        raise ValueError(f'Scored candidates require columns: {sorted(required)}')
    candidates = scored_candidates.copy()
    if not CANDIDATE_POLICIES[policy]['exclude_seen']:
        return candidates
    seen = (
        early_history.loc[early_history['in_candidate_catalog'], ['learner_id', 'item_id']]
        .drop_duplicates().assign(_seen=True)
    )
    candidates = candidates.merge(seen, on=['learner_id', 'item_id'], how='left')
    return candidates[candidates['_seen'].ne(True)].drop(columns='_seen')

def assert_candidate_subset(scored_candidates, catalog):
    candidate_items = set(scored_candidates['item_id'])
    catalog_items = set(catalog['item_id'])
    unknown = candidate_items - catalog_items
    assert not unknown, f'{len(unknown)} scored items are outside the candidate catalogue.'


## Step 5 — Ranking, coverage and cold-start metrics

Only learners with at least one in-catalog relevant future item are evaluable. Incomplete recommendation lists are penalised using K as the Precision@K denominator.

In [6]:
def metrics_for_user(recommended_items, relevant_items, k):
    recommended = list(recommended_items[:k])
    relevant = set(relevant_items)
    if not relevant:
        return None
    hits = np.array([item in relevant for item in recommended], dtype=float)
    padded_hits = np.pad(hits, (0, max(0, k - len(hits))))[:k]
    hit_count = padded_hits.sum()
    precision = hit_count / k
    recall = hit_count / len(relevant)
    discounts = np.log2(np.arange(2, k + 2))
    dcg = np.sum(padded_hits / discounts)
    ideal_length = min(len(relevant), k)
    idcg = np.sum(np.ones(ideal_length) / discounts[:ideal_length])
    ndcg = dcg / idcg if idcg else 0.0
    hit_positions = np.flatnonzero(padded_hits)
    average_precision = (
        sum(padded_hits[:position + 1].sum() / (position + 1) for position in hit_positions)
        / min(len(relevant), k)
    )
    return {
        'Precision': precision, 'Recall': recall, 'NDCG': ndcg,
        'MAP': average_precision, 'HitRate': float(hit_count > 0),
    }

def evaluate_recommendations(
    recommendations, relevance, catalog, learner_splits,
    relevance_column, evaluable_column, cold_start_column,
    dataset, task, candidate_policy, relevance_name, ks=METRIC_KS,
):
    started_at = time.perf_counter()
    required_recommendation_columns = {'learner_id', 'item_id', 'score'}
    if not required_recommendation_columns.issubset(recommendations.columns):
        raise ValueError('Recommendations require learner_id, item_id and score.')
    if candidate_policy not in CANDIDATE_POLICIES:
        raise ValueError(f'Unknown candidate policy: {candidate_policy}')
    if relevance_column not in relevance.columns:
        raise ValueError(f'Missing relevance column: {relevance_column}')
    assert not recommendations.duplicated(['learner_id', 'item_id']).any()
    assert np.isfinite(pd.to_numeric(recommendations['score'], errors='coerce')).all()
    assert_candidate_subset(recommendations, catalog)

    ranked = recommendations.sort_values(
        ['learner_id', 'score', 'item_id'], ascending=[True, False, True],
        kind='mergesort',
    )
    relevance_mask = (
        relevance['in_candidate_catalog'] & relevance[relevance_column].eq(1)
    )
    if candidate_policy == 'novel_only':
        if 'seen_in_early' not in relevance.columns:
            raise ValueError('novel_only evaluation requires seen_in_early labels.')
        relevance_mask &= ~relevance['seen_in_early'].fillna(False).astype(bool)
    relevant = relevance.loc[
        relevance_mask, ['learner_id', 'item_id']
    ].drop_duplicates()
    relevant_by_user = relevant.groupby('learner_id')['item_id'].agg(set).to_dict()
    ranked_by_user = ranked.groupby('learner_id')['item_id'].agg(list).to_dict()

    validation = learner_splits[learner_splits['cohort'].eq('validation')].copy()
    eligible_users = set(validation.loc[validation[evaluable_column], 'learner_id'])
    eligible_users &= set(relevant_by_user)
    segments = {
        'all': eligible_users,
        'cold_start': eligible_users & set(validation.loc[validation[cold_start_column], 'learner_id']),
        'non_cold_start': eligible_users & set(validation.loc[~validation[cold_start_column], 'learner_id']),
    }

    rows = []
    catalog_size = len(catalog)
    for segment_name, users in segments.items():
        if not users:
            continue
        for k in ks:
            user_rows = []
            recommended_union = set()
            list_lengths = []
            for learner_id in sorted(users):
                recommended = ranked_by_user.get(learner_id, [])[:k]
                values = metrics_for_user(recommended, relevant_by_user[learner_id], k)
                user_rows.append(values)
                recommended_union.update(recommended)
                list_lengths.append(len(recommended))
            user_metrics = pd.DataFrame(user_rows)
            rows.append({
                'Segment': segment_name, 'K': k,
                'PrecisionAtK': user_metrics['Precision'].mean(),
                'RecallAtK': user_metrics['Recall'].mean(),
                'NDCGAtK': user_metrics['NDCG'].mean(),
                'MAPAtK': user_metrics['MAP'].mean(),
                'HitRateAtK': user_metrics['HitRate'].mean(),
                'CatalogCoverageAtK': len(recommended_union) / catalog_size,
                'MeanRecommendations': float(np.mean(list_lengths)),
                'EvaluatedLearners': len(users),
                'EvaluableRate': len(eligible_users) / len(validation),
            })
    result = pd.DataFrame(rows)
    result.insert(0, 'RelevanceDefinition', relevance_name)
    result.insert(0, 'CandidatePolicy', candidate_policy)
    result.insert(0, 'Task', task)
    result.insert(0, 'Dataset', dataset)
    result['RecommendationMemoryBytes'] = int(recommendations.memory_usage(deep=True).sum())
    result['MetricRuntimeSeconds'] = time.perf_counter() - started_at
    return result

## Save and verify the Phase 3 steps 1–5 setup

These are setup artifacts, not recommendation results. Baseline result files are created only when steps 6 onward are implemented.

In [7]:
contract_validation.to_csv(OUTPUT_ROOT / 'phase3_contract_validation.csv', index=False)
task_table.to_csv(OUTPUT_ROOT / 'phase3_task_definitions.csv', index=False)
phase3_setup_config = {
    'implemented_steps': [1, 2, 3, 4, 5],
    'pending_steps': list(range(6, 17)),
    'benchmark_root': str(BENCHMARK_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'metric_ks': list(METRIC_KS),
    'primary_relevance': PRIMARY_RELEVANCE,
    'relevance_definitions': RELEVANCE_DEFINITIONS,
    'candidate_policies': CANDIDATE_POLICIES,
    'primary_task': 'ASSISTments problem recommendation',
    'dense_cross_product_forbidden': True,
    'validation_future_role': 'evaluation_only',
    'random_state': RANDOM_STATE,
}
with open(OUTPUT_ROOT / 'phase3_setup_config.json', 'w', encoding='utf-8') as file:
    json.dump(phase3_setup_config, file, indent=2)

setup_files = [
    OUTPUT_ROOT / 'phase3_contract_validation.csv',
    OUTPUT_ROOT / 'phase3_task_definitions.csv',
    OUTPUT_ROOT / 'phase3_setup_config.json',
]
setup_manifest = pd.DataFrame([
    {'File': path.name, 'Bytes': path.stat().st_size}
    for path in setup_files
])
setup_manifest.to_csv(OUTPUT_ROOT / 'phase3_setup_manifest.csv', index=False)
assert len(pd.read_csv(OUTPUT_ROOT / 'phase3_contract_validation.csv')) == 18
assert len(pd.read_csv(OUTPUT_ROOT / 'phase3_task_definitions.csv')) == 4
assert len(pd.read_csv(OUTPUT_ROOT / 'phase3_setup_manifest.csv')) == 3
display(setup_manifest)

,File,Bytes
0,phase3_contract_validation.csv,1084
1,phase3_task_definitions.csv,1028
2,phase3_setup_config.json,1022


## Step 6 — Popularity baselines

Two deterministic global baselines are compared for every dataset/task/relevance combination:

- `popularity_discovery_early`: discovery-early interaction counts from the frozen candidate catalogue.
- `popularity_discovery_future`: the number of discovery learners with the relevant future item. Validation-future labels are excluded before scores are fitted.

Recommendations are generated directly as sparse top-20 rows. For `novel_only`, the ranked catalogue is scanned until each learner has up to 20 unseen items; no dense learner-by-item matrix is constructed.

In [8]:
MAX_RECOMMENDATIONS = max(METRIC_KS)

def load_task_tables(task_definition):
    directory = DATASET_DIRS[task_definition['Dataset']]
    return {
        'learners': pd.read_parquet(directory / 'learner_splits.parquet'),
        'catalog': pd.read_parquet(directory / task_definition['CatalogFile']),
        'early_history': pd.read_parquet(directory / task_definition['EarlyHistoryFile']),
        'future_relevance': pd.read_parquet(directory / task_definition['FutureRelevanceFile']),
    }

def top_k_sparse(scored_candidates, max_k=MAX_RECOMMENDATIONS):
    required = {'learner_id', 'item_id', 'score'}
    if not required.issubset(scored_candidates.columns):
        raise ValueError(f'Scored candidates require columns: {sorted(required)}')
    if scored_candidates.empty:
        return pd.DataFrame(columns=['learner_id', 'item_id', 'score', 'rank'])
    assert not scored_candidates.duplicated(['learner_id', 'item_id']).any()
    ranked = scored_candidates[['learner_id', 'item_id', 'score']].sort_values(
        ['learner_id', 'score', 'item_id'],
        ascending=[True, False, True],
        kind='mergesort',
    )
    ranked = ranked.groupby('learner_id', sort=False).head(max_k).copy()
    ranked['rank'] = ranked.groupby('learner_id', sort=False).cumcount() + 1
    return ranked

def global_top_k_recommendations(
    validation_learners, item_scores, early_history, candidate_policy,
    max_k=MAX_RECOMMENDATIONS,
):
    if candidate_policy not in CANDIDATE_POLICIES:
        raise ValueError(f'Unknown candidate policy: {candidate_policy}')
    required = {'item_id', 'score'}
    if not required.issubset(item_scores.columns):
        raise ValueError('Item scores require item_id and score.')
    assert not item_scores['item_id'].duplicated().any()
    assert np.isfinite(pd.to_numeric(item_scores['score'], errors='coerce')).all()
    ranking = list(
        item_scores[['item_id', 'score']]
        .sort_values(['score', 'item_id'], ascending=[False, True], kind='mergesort')
        .itertuples(index=False, name=None)
    )
    learner_ids = sorted(pd.Series(validation_learners).astype(str).unique())
    seen_by_learner = {}
    if candidate_policy == 'novel_only':
        validation_set = set(learner_ids)
        seen = early_history[
            early_history['in_candidate_catalog']
            & early_history['learner_id'].astype(str).isin(validation_set)
        ][['learner_id', 'item_id']].drop_duplicates()
        seen_by_learner = seen.groupby('learner_id')['item_id'].agg(set).to_dict()

    rows = []
    for learner_id in learner_ids:
        excluded = seen_by_learner.get(learner_id, set())
        rank = 0
        for item_id, score in ranking:
            if item_id in excluded:
                continue
            rank += 1
            rows.append((learner_id, item_id, float(score), rank))
            if rank == max_k:
                break
    recommendations = pd.DataFrame(
        rows, columns=['learner_id', 'item_id', 'score', 'rank']
    )
    assert not recommendations.duplicated(['learner_id', 'item_id']).any()
    assert recommendations.groupby('learner_id').size().le(max_k).all()
    return recommendations

def discovery_early_popularity_scores(catalog):
    scores = catalog[
        ['item_id', 'training_interactions', 'training_learners', 'training_popularity']
    ].copy()
    scores['score'] = scores['training_interactions'].astype(float)
    scores['normalised_score'] = scores['training_popularity'].astype(float)
    scores['score_source'] = 'discovery_early_interactions'
    return scores

def discovery_future_popularity_scores(
    relevance, learners, catalog, relevance_column,
):
    discovery_ids = set(
        learners.loc[learners['cohort'].eq('discovery'), 'learner_id'].astype(str)
    )
    validation_ids = set(
        learners.loc[learners['cohort'].eq('validation'), 'learner_id'].astype(str)
    )
    training_labels = relevance[
        relevance['learner_id'].astype(str).isin(discovery_ids)
        & relevance['in_candidate_catalog']
    ][['learner_id', 'item_id', relevance_column]].copy()
    assert set(training_labels['learner_id']).issubset(discovery_ids)
    assert set(training_labels['learner_id']).isdisjoint(validation_ids)
    assert not training_labels.duplicated(['learner_id', 'item_id']).any()
    relevant_labels = training_labels[training_labels[relevance_column].eq(1)]
    counts = (
        relevant_labels.groupby('item_id')['learner_id']
        .nunique().rename('discovery_relevant_learners').reset_index()
    )
    scores = catalog[['item_id']].merge(counts, on='item_id', how='left')
    scores['discovery_relevant_learners'] = (
        scores['discovery_relevant_learners'].fillna(0).astype('int64')
    )
    scores['score'] = scores['discovery_relevant_learners'].astype(float)
    total = scores['score'].sum()
    scores['normalised_score'] = scores['score'] / total if total else 0.0
    scores['score_source'] = f'discovery_future_{relevance_column}'
    return scores

In [9]:
popularity_metric_parts = []
popularity_recommendation_parts = []
popularity_score_parts = []

for task_definition in TASK_DEFINITIONS:
    dataset = task_definition['Dataset']
    task = task_definition['Task']
    tables = load_task_tables(task_definition)
    learners = tables['learners']
    catalog = tables['catalog']
    early_history = tables['early_history']
    future_relevance = tables['future_relevance']
    validation_ids = learners.loc[
        learners['cohort'].eq('validation'), 'learner_id'
    ].astype(str)

    early_scores = discovery_early_popularity_scores(catalog)
    assert_candidate_subset(early_scores[['item_id', 'score']], catalog)

    for relevance_name, relevance_column in RELEVANCE_DEFINITIONS.items():
        future_scores = discovery_future_popularity_scores(
            future_relevance, learners, catalog, relevance_column
        )
        models = {
            'popularity_discovery_early': early_scores,
            'popularity_discovery_future': future_scores,
        }
        for model_name, item_scores in models.items():
            score_output = item_scores.copy()
            score_output.insert(0, 'RelevanceDefinition', relevance_name)
            score_output.insert(0, 'Model', model_name)
            score_output.insert(0, 'Task', task)
            score_output.insert(0, 'Dataset', dataset)
            popularity_score_parts.append(score_output)

            for candidate_policy in CANDIDATE_POLICIES:
                recommendation_started = time.perf_counter()
                recommendations = global_top_k_recommendations(
                    validation_ids, item_scores, early_history,
                    candidate_policy, MAX_RECOMMENDATIONS,
                )
                assert_candidate_subset(recommendations, catalog)
                recommendation_runtime = time.perf_counter() - recommendation_started
                metrics = evaluate_recommendations(
                    recommendations, future_relevance, catalog, learners,
                    relevance_column, task_definition['EvaluableColumn'],
                    task_definition['ColdStartColumn'], dataset, task,
                    candidate_policy, relevance_name,
                )
                metrics.insert(4, 'Model', model_name)
                metrics['RecommendationBuildRuntimeSeconds'] = recommendation_runtime
                popularity_metric_parts.append(metrics)

                recommendation_output = recommendations.copy()
                recommendation_output.insert(0, 'RelevanceDefinition', relevance_name)
                recommendation_output.insert(0, 'CandidatePolicy', candidate_policy)
                recommendation_output.insert(0, 'Model', model_name)
                recommendation_output.insert(0, 'Task', task)
                recommendation_output.insert(0, 'Dataset', dataset)
                popularity_recommendation_parts.append(recommendation_output)

popularity_metrics = pd.concat(popularity_metric_parts, ignore_index=True)
popularity_recommendations = pd.concat(
    popularity_recommendation_parts, ignore_index=True
)
popularity_scores = pd.concat(popularity_score_parts, ignore_index=True)
assert set(popularity_metrics['Model']) == {
    'popularity_discovery_early', 'popularity_discovery_future'
}
assert set(popularity_metrics['CandidatePolicy']) == set(CANDIDATE_POLICIES)
assert set(popularity_metrics['RelevanceDefinition']) == set(RELEVANCE_DEFINITIONS)
display(popularity_metrics.sort_values(
    ['Dataset', 'Task', 'RelevanceDefinition', 'CandidatePolicy', 'Model', 'Segment', 'K']
))

,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,MAPAtK,HitRateAtK,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,EvaluableRate,RecommendationMemoryBytes,MetricRuntimeSeconds,RecommendationBuildRuntimeSeconds
0,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,all,5,0.011341,0.003534,0.011477,0.009014,0.021014,0.000126,5.000000,5996,0.899355,17866552,3.039687,0.310391
1,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,all,10,0.011474,0.007234,0.011900,0.008486,0.027352,0.000251,10.000000,5996,0.899355,17866552,3.039687,0.310391
2,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,all,20,0.011116,0.013276,0.014485,0.009746,0.032188,0.000503,20.000000,5996,0.899355,17866552,3.039687,0.310391
3,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,cold_start,5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000126,5.000000,21,0.899355,17866552,3.039687,0.310391
4,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,cold_start,10,0.000000,0.000000,0.000000,0.000000,0.000000,0.000251,10.000000,21,0.899355,17866552,3.039687,0.310391
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,KDD,skill,novel_only,successful,popularity_discovery_future,all,10,0.294318,0.597980,0.509490,0.402189,0.818182,0.808081,9.965909,88,0.778761,462925,0.063961,0.012730
236,KDD,skill,novel_only,successful,popularity_discovery_future,all,20,0.204545,0.783098,0.566299,0.434034,0.931818,0.929293,19.715909,88,0.778761,462925,0.063961,0.012730
237,KDD,skill,novel_only,successful,popularity_discovery_future,non_cold_start,5,0.377273,0.382336,0.452428,0.384201,0.715909,0.585859,5.000000,88,0.778761,462925,0.063961,0.012730
238,KDD,skill,novel_only,successful,popularity_discovery_future,non_cold_start,10,0.294318,0.597980,0.509490,0.402189,0.818182,0.808081,9.965909,88,0.778761,462925,0.063961,0.012730


## Step 7 — Weak-skill baseline

This baseline applies only to skill recommendation. It scores validation learners' supported `weak` and `developing` skills as `(1 - empirical_bayes_mastery) * mastery_evidence_confidence`.

`insufficient_evidence` remains a separate diagnostic state and is never interpreted as demonstrated weakness. The model has no unobserved-skill transfer signal, so `novel_only` correctly produces empty recommendation lists instead of adding an undeclared popularity backfill.

In [10]:
WEAK_SKILL_STATES = {'weak', 'developing'}
weak_skill_metric_parts = []
weak_skill_recommendation_parts = []
weak_skill_score_parts = []
weak_skill_diagnostic_parts = []

for task_definition in [row for row in TASK_DEFINITIONS if row['Task'] == 'skill']:
    dataset = task_definition['Dataset']
    tables = load_task_tables(task_definition)
    learners = tables['learners']
    catalog = tables['catalog']
    early_history = tables['early_history']
    future_relevance = tables['future_relevance']
    validation_ids = set(
        learners.loc[learners['cohort'].eq('validation'), 'learner_id'].astype(str)
    )
    validation_history = early_history[
        early_history['learner_id'].astype(str).isin(validation_ids)
        & early_history['in_candidate_catalog']
    ].copy()

    diagnostics = (
        validation_history.groupby('skill_state', dropna=False)
        .size().rename('LearnerSkillRows').reset_index()
    )
    diagnostics.insert(0, 'Dataset', dataset)
    diagnostics['UsedAsWeaknessEvidence'] = diagnostics['skill_state'].isin(
        WEAK_SKILL_STATES
    )
    weak_skill_diagnostic_parts.append(diagnostics)

    scored = validation_history[
        validation_history['skill_state'].isin(WEAK_SKILL_STATES)
    ][
        ['learner_id', 'item_id', 'empirical_bayes_mastery',
         'mastery_evidence_confidence', 'skill_state']
    ].copy()
    assert not scored['skill_state'].eq('insufficient_evidence').any()
    assert scored['empirical_bayes_mastery'].between(0, 1).all()
    assert scored['mastery_evidence_confidence'].between(0, 1).all()
    scored['score'] = (
        (1.0 - scored['empirical_bayes_mastery'])
        * scored['mastery_evidence_confidence']
    )
    assert np.isfinite(scored['score']).all()
    assert_candidate_subset(scored, catalog)

    score_output = scored.copy()
    score_output.insert(0, 'Model', 'weak_skill_mastery_confidence')
    score_output.insert(0, 'Task', 'skill')
    score_output.insert(0, 'Dataset', dataset)
    weak_skill_score_parts.append(score_output)

    for candidate_policy in CANDIDATE_POLICIES:
        recommendation_started = time.perf_counter()
        policy_scores = apply_candidate_policy(
            scored, validation_history, candidate_policy
        )
        recommendations = top_k_sparse(policy_scores, MAX_RECOMMENDATIONS)
        if candidate_policy == 'novel_only':
            assert recommendations.empty
        recommendation_runtime = time.perf_counter() - recommendation_started
        assert_candidate_subset(recommendations, catalog)

        for relevance_name, relevance_column in RELEVANCE_DEFINITIONS.items():
            metrics = evaluate_recommendations(
                recommendations, future_relevance, catalog, learners,
                relevance_column, task_definition['EvaluableColumn'],
                task_definition['ColdStartColumn'], dataset, 'skill',
                candidate_policy, relevance_name,
            )
            metrics.insert(4, 'Model', 'weak_skill_mastery_confidence')
            metrics['RecommendationBuildRuntimeSeconds'] = recommendation_runtime
            metrics['PolicyInterpretation'] = np.where(
                metrics['CandidatePolicy'].eq('novel_only'),
                'no_unseen_transfer_signal', 'personalised_weakness_ranking',
            )
            weak_skill_metric_parts.append(metrics)

            recommendation_output = recommendations.copy()
            recommendation_output.insert(0, 'RelevanceDefinition', relevance_name)
            recommendation_output.insert(0, 'CandidatePolicy', candidate_policy)
            recommendation_output.insert(0, 'Model', 'weak_skill_mastery_confidence')
            recommendation_output.insert(0, 'Task', 'skill')
            recommendation_output.insert(0, 'Dataset', dataset)
            weak_skill_recommendation_parts.append(recommendation_output)

weak_skill_metrics = pd.concat(weak_skill_metric_parts, ignore_index=True)
weak_skill_recommendations = pd.concat(
    weak_skill_recommendation_parts, ignore_index=True
)
weak_skill_scores = pd.concat(weak_skill_score_parts, ignore_index=True)
weak_skill_diagnostics = pd.concat(
    weak_skill_diagnostic_parts, ignore_index=True
)
assert not weak_skill_diagnostics.loc[
    weak_skill_diagnostics['skill_state'].eq('insufficient_evidence'),
    'UsedAsWeaknessEvidence',
].any()
display(weak_skill_metrics.sort_values(
    ['Dataset', 'RelevanceDefinition', 'CandidatePolicy', 'Segment', 'K']
))
display(weak_skill_diagnostics.sort_values(['Dataset', 'skill_state']))

/tmp/ipykernel_700/1798824526.py:89: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  weak_skill_recommendations = pd.concat(


,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,MAPAtK,HitRateAtK,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,EvaluableRate,RecommendationMemoryBytes,MetricRuntimeSeconds,RecommendationBuildRuntimeSeconds,PolicyInterpretation
0,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,all,5,0.244773,0.197409,0.321089,0.247638,0.616012,0.844720,3.291688,3922,0.588271,2980604,1.636041,0.016420,personalised_weakness_ranking
1,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,all,10,0.168256,0.227048,0.295214,0.212911,0.636410,0.869565,4.569862,3922,0.588271,2980604,1.636041,0.016420,personalised_weakness_ranking
2,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,all,20,0.097081,0.238813,0.275014,0.190726,0.639215,0.875776,5.279704,3922,0.588271,2980604,1.636041,0.016420,personalised_weakness_ranking
3,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,cold_start,5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,133,0.588271,2980604,1.636041,0.016420,personalised_weakness_ranking
4,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,cold_start,10,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,133,0.588271,2980604,1.636041,0.016420,personalised_weakness_ranking
5,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,cold_start,20,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,133,0.588271,2980604,1.636041,0.016420,personalised_weakness_ranking
6,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,non_cold_start,5,0.253365,0.204338,0.332360,0.256330,0.637635,0.844720,3.407231,3789,0.588271,2980604,1.636041,0.016420,personalised_weakness_ranking
7,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,non_cold_start,10,0.174162,0.235017,0.305576,0.220385,0.658749,0.869565,4.730272,3789,0.588271,2980604,1.636041,0.016420,personalised_weakness_ranking
8,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,non_cold_start,20,0.100488,0.247196,0.284667,0.197421,0.661652,0.875776,5.465030,3789,0.588271,2980604,1.636041,0.016420,personalised_weakness_ranking
18,ASSISTments,skill,novel_only,attempted,weak_skill_mastery_confidence,all,5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3441,0.516124,132,1.238312,0.027202,no_unseen_transfer_signal


,Dataset,skill_state,LearnerSkillRows,UsedAsWeaknessEvidence
0,ASSISTments,developing,14898,True
1,ASSISTments,insufficient_evidence,17882,False
2,ASSISTments,mastered,12039,False
3,ASSISTments,weak,6963,True
4,KDD,developing,1001,True
5,KDD,insufficient_evidence,500,False
6,KDD,mastered,1175,False
7,KDD,weak,1094,True


## Save and verify Steps 6–7 artifacts

These are staged baseline artifacts. The consolidated Phase 3 filenames required by Step 15 will be produced only after all baseline families are implemented.

In [11]:
steps_6_7_metrics = pd.concat(
    [popularity_metrics, weak_skill_metrics], ignore_index=True, sort=False
)
steps_6_7_recommendations = pd.concat(
    [popularity_recommendations, weak_skill_recommendations],
    ignore_index=True, sort=False,
)

steps_6_7_paths = {
    'metrics': OUTPUT_ROOT / 'steps_6_7_metrics.csv',
    'recommendations': OUTPUT_ROOT / 'steps_6_7_recommendations.parquet',
    'popularity_scores': OUTPUT_ROOT / 'step6_popularity_scores.parquet',
    'weak_skill_scores': OUTPUT_ROOT / 'step7_weak_skill_scores.parquet',
    'weak_skill_diagnostics': OUTPUT_ROOT / 'step7_weak_skill_diagnostics.csv',
    'config': OUTPUT_ROOT / 'steps_6_7_config.json',
}
steps_6_7_metrics.to_csv(steps_6_7_paths['metrics'], index=False)
steps_6_7_recommendations.to_parquet(
    steps_6_7_paths['recommendations'], index=False, compression='snappy'
)
popularity_scores.to_parquet(
    steps_6_7_paths['popularity_scores'], index=False, compression='snappy'
)
weak_skill_scores.to_parquet(
    steps_6_7_paths['weak_skill_scores'], index=False, compression='snappy'
)
weak_skill_diagnostics.to_csv(
    steps_6_7_paths['weak_skill_diagnostics'], index=False
)

steps_6_7_config = {
    'implemented_steps': [6, 7],
    'metric_ks': list(METRIC_KS),
    'maximum_saved_recommendations_per_learner': MAX_RECOMMENDATIONS,
    'popularity_models': [
        'popularity_discovery_early', 'popularity_discovery_future'
    ],
    'discovery_future_training_learners_only': True,
    'validation_future_role': 'evaluation_only',
    'candidate_policies': list(CANDIDATE_POLICIES),
    'relevance_definitions': RELEVANCE_DEFINITIONS,
    'weak_skill_model': 'weak_skill_mastery_confidence',
    'weak_skill_formula': '(1 - empirical_bayes_mastery) * mastery_evidence_confidence',
    'weak_skill_states': sorted(WEAK_SKILL_STATES),
    'insufficient_evidence_handling': 'diagnostic_only_not_weakness',
    'weak_skill_novel_only_handling': 'empty_no_unseen_transfer_signal',
    'dense_learner_item_matrix_constructed': False,
}
with open(steps_6_7_paths['config'], 'w', encoding='utf-8') as file:
    json.dump(steps_6_7_config, file, indent=2)

steps_6_7_manifest = pd.DataFrame([
    {
        'Artifact': name,
        'File': path.name,
        'Rows': (
            pq.ParquetFile(path).metadata.num_rows
            if path.suffix == '.parquet'
            else (len(pd.read_csv(path)) if path.suffix == '.csv' else 1)
        ),
        'Bytes': path.stat().st_size,
    }
    for name, path in steps_6_7_paths.items()
])
steps_6_7_manifest.to_csv(
    OUTPUT_ROOT / 'steps_6_7_artifact_manifest.csv', index=False
)
assert len(steps_6_7_manifest) == 6
assert pq.ParquetFile(
    steps_6_7_paths['recommendations']
).metadata.num_rows == len(steps_6_7_recommendations)
assert len(pd.read_csv(steps_6_7_paths['metrics'])) == len(steps_6_7_metrics)
display(steps_6_7_manifest)

,Artifact,File,Rows,Bytes
0,metrics,steps_6_7_metrics.csv,300,79261
1,recommendations,steps_6_7_recommendations.parquet,2215406,1109217
2,popularity_scores,step6_popularity_scores.parquet,163576,1193533
3,weak_skill_scores,step7_weak_skill_scores.parquet,23956,339990
4,weak_skill_diagnostics,step7_weak_skill_diagnostics.csv,8,304
5,config,steps_6_7_config.json,1,909
